# WIF3009 Capstone: Autonomous Community Manager Agent
**Cluster 03:** Language, Chat & Networks
**Project 19:** Discord Moderation Bot

## 1. Executive Summary
Digital communities require constant moderation to prevent toxicity spirals. This project bridges backend infrastructure and continuous NLP tracking to act as an autonomous community manager. It executes longitudinal sentiment tracking to trigger rolling-window interventions.

## 2. Technical Architecture
* **Bot Framework:** `discord.py` (Asynchronous event processing)
* **Storage:** SQLite (Prototyping) -> PostgreSQL (Production Deployment)
* **NLP Engine:** OpenRouter (Open-source reasoning models)
* **Trigger Logic:** Statistical Process Control (SPC) moving averages

---
## 3. Implementation Code

In [1]:
# Install the required libraries for our stack
!pip install discord.py python-dotenv aiohttp nest_asyncio pandas asyncpg sqlalchemy psycopg2-binary transformers torch

import discord
from discord.ext import commands
import nest_asyncio
import pandas as pd
import aiohttp
import json
import os
from dotenv import load_dotenv

# Apply this to allow discord.py to run inside Jupyter without crashing
nest_asyncio.apply()

print("✅ Dependencies installed and imported successfully.")

✅ Dependencies installed and imported successfully.


In [2]:
import asyncpg
import asyncio
import datetime
import os
from dotenv import load_dotenv

# Securely load the hidden variables
load_dotenv()

# Pull the URL from the .env file instead of hardcoding it!
DATABASE_URL = os.getenv("SUPABASE_URL")

class DatabaseManager:
    # ... (keep the rest of your class exactly the same!)
    def __init__(self):
        self.pool = None

    async def connect(self):
        self.pool = await asyncpg.create_pool(dsn=DATABASE_URL)
        await self._init_schema()
        print("☁️ Supabase Cloud Connection Established.")

    async def _init_schema(self):
        async with self.pool.acquire() as conn:
            # 1. User Reputation Table
            await conn.execute('''
                CREATE TABLE IF NOT EXISTS users (
                    user_id TEXT PRIMARY KEY,
                    trust_score REAL DEFAULT 100.0,
                    last_active TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                );
            ''')
            # 2. Core Messages Table
            await conn.execute('''
                CREATE TABLE IF NOT EXISTS messages (
                    message_id TEXT PRIMARY KEY,
                    user_id TEXT REFERENCES users(user_id),
                    channel_id TEXT,
                    content TEXT,
                    toxicity_score REAL,
                    timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP
                );
            ''')
            print("✅ Cloud Database Schema Verified.")

    async def log_message(self, message_id, user_id, channel_id, content, toxicity_score):
        async with self.pool.acquire() as conn:
            # Update user activity
            await conn.execute('''
                INSERT INTO users (user_id, last_active) 
                VALUES ($1, $2)
                ON CONFLICT (user_id) DO UPDATE SET last_active = $2
            ''', str(user_id), datetime.datetime.now())
            # Log the message
            await conn.execute('''
                INSERT INTO messages (message_id, user_id, channel_id, content, toxicity_score, timestamp)
                VALUES ($1, $2, $3, $4, $5, $6)
            ''', str(message_id), str(user_id), str(channel_id), content, toxicity_score, datetime.datetime.now())

    async def update_trust_score(self, user_id, penalty_amount):
        async with self.pool.acquire() as conn:
            await conn.execute('''
                UPDATE users SET trust_score = trust_score - $1 WHERE user_id = $2
            ''', penalty_amount, str(user_id))
    
    async def reward_trust_score(self, user_id, reward_amount=1.0):
                """Rewards trust points for good behavior, capped at a maximum of 100.0"""
                async with self.pool.acquire() as conn:
                    await conn.execute('''
                        UPDATE users 
                        SET trust_score = LEAST(trust_score + $1, 100.0) 
                        WHERE user_id = $2
                    ''', reward_amount, str(user_id))

    async def get_recent_scores(self, channel_id, limit=5):
        async with self.pool.acquire() as conn:
            records = await conn.fetch('''
                SELECT toxicity_score FROM messages 
                WHERE channel_id = $1 ORDER BY timestamp DESC LIMIT $2
            ''', str(channel_id), limit)
            return [record['toxicity_score'] for record in records]
            

# Initialize and connect the manager
db = DatabaseManager()
asyncio.run(db.connect())

✅ Cloud Database Schema Verified.
☁️ Supabase Cloud Connection Established.


In [3]:
import asyncio
from transformers import pipeline

print("⏳ Initializing local NLP Toxicity Model pipeline...")
try:
    local_toxicity_analyzer = pipeline(
        "text-classification", 
        model="martin-ha/toxic-comment-model"
    )
    print("✅ Local NLP Toxicity Engine Ready! Operating in Model-First validation mode.")
except Exception as e:
    print(f"⚠️ Failed to load local model: {e}")
    local_toxicity_analyzer = None

async def get_toxicity_score(text):
    """
    Evaluates text toxicity using the local NLP Transformer pipeline first.
    Applies structural array inspection and deterministic corrections if misclassified.
    """
    if local_toxicity_analyzer is None:
        return 0.0
        
    try:
        # 1. RUN PRIMARY MACHINE LEARNING INFERENCE
        loop = asyncio.get_event_loop()
        predictions = await loop.run_in_executor(None, local_toxicity_analyzer, text)
        
        # Normalize text strings for fallback keyword validation
        clean_text = text.lower().strip()
        normalized = clean_text.replace(" ", "").replace("-", "")
        severe_terms = [
            "retard", "bodo", "bapakkau", "anjing", "pukimak", "bitch", 
            "nigga", "nigger", "killu", "killurself", "worthless", "dipwit", "stfu"
        ]
        has_severe_keyword = any(term in normalized for term in severe_terms) or "kill yourself" in clean_text

        # 2. EVALUATE STRUCTURAL ARRAY OUTPUT
        if predictions and len(predictions) > 0:
            result = predictions[0]
            label = result.get("label", "").lower()
            confidence = float(result.get("score", 0.0))
            
            # Scenario A: Model correctly identifies toxicity
            if label == "toxic":
                return float(confidence)
                
            # Scenario B: Model says 'non-toxic' but text contains explicit high-severity terms
            if label == "non-toxic" and has_severe_keyword:
                # Calculate the mathematical inverse of its non-toxic confidence
                inverse_score = 1.0 - confidence
                
                # If inverse math is too weak, force a hard fallback correction score
                fallback_score = max(0.95, inverse_score)
                print(f"[AI Correction Active] Model misclassified severe text. Raw Output: {predictions}. Recalculated Score: {fallback_score:.2f}")
                return float(fallback_score)
            
            # Scenario C: Model says 'non-toxic' and text is genuinely normal chat
            if label == "non-toxic":
                # Inverse calculation accounts for mild micro-toxicity spikes
                return float(1.0 - confidence)
                
        return 0.0
    except Exception as e:
        print(f"⚠️ Local Scoring Exception: {e}")
        return 0.0

c:\Users\user\anaconda3\envs\wif3009\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


⏳ Initializing local NLP Toxicity Model pipeline...


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 6010.27it/s]


✅ Local NLP Toxicity Engine Ready! Operating in Model-First validation mode.


In [ ]:
import time
import pandas as pd
import discord
from discord.ext import commands
import os
from dotenv import load_dotenv

load_dotenv()
DISCORD_TOKEN = os.getenv("DISCORD_BOT_TOKEN")

# Tracks when the bot last issued an intervention per channel
channel_cooldowns = {}

# 💡 LOCAL WINDOW CACHE: Wipes the memory hangover instantly
channel_windows = {}

intents = discord.Intents.default()
intents.message_content = True
bot = commands.Bot(command_prefix="!", intents=intents)

@bot.event
async def on_ready():
    print(f'✅ Aegis Agent Online: Logged in as {bot.user}')
    print('--- AI Core & Real-Time Telemetry Loop Active ---')

@bot.event
async def on_message(message):
    # Never evaluate the bot's own text strings
    if message.author == bot.user:
        return

    try:
        # 1. OBTAIN AND VALIDATE TELEMETRY METRIC
        score = await get_toxicity_score(message.content)
        
        # Fault-tolerant database payload logging (Keeps working in background)
        try:
            await db.log_message(message.id, message.author.id, message.channel.id, message.content, score)
        except Exception as db_err:
            print(f"⚠️ Database Sync Delay/Failure (PostgreSQL Connection): {db_err}")

        print(f"[Telemetry Scan] Score: {score:.2f} | {message.author}: {message.content}")

        # REPUTATION REWARDS
        if score < 0.20:
            try:
                await db.reward_trust_score(message.author.id, 1.0)
            except:
                pass

        # Initialize the channel's local window if it doesn't exist yet
        if message.channel.id not in channel_windows:
            channel_windows[message.channel.id] = []
            
        # Append the clean float score to our local tracker
        channel_windows[message.channel.id].append(score)
        
        # Maintain a strict rolling window size of 5
        if len(channel_windows[message.channel.id]) > 5:
            channel_windows[message.channel.id].pop(0)

        # 2. --- REAL-TIME STATISTICAL PROCESS CONTROL (SPC) ---
        recent_scores = channel_windows[message.channel.id]
        
        if len(recent_scores) == 5:
            df = pd.DataFrame(recent_scores, columns=['score'])
            
            moving_avg = df['score'].mean()
            std_deviation = df['score'].std()
            variance = df['score'].var()
            
            # Optimized scaling formulas prevent zero-inflation from locking up your alerts
            adjustment = (0.20 * std_deviation) if not pd.isna(std_deviation) else 0.02
            dynamic_ucl = min(0.55, 0.40 + adjustment) 
            
            print(f"SPC Metrics -> Mean: {moving_avg:.2f} | StdDev: {std_deviation:.2f} | Var: {variance:.2f} | Dynamic Target UCL: {dynamic_ucl:.2f}")
            
            # 3. INTERVENTION EVALUATION TRIGGER
            if moving_avg >= dynamic_ucl:
                last_warning = channel_cooldowns.get(message.channel.id, 0)
                
                if time.time() - last_warning > 60:
                    warning_msg = f"⚠️ **Aegis Automated Intervention** ⚠️\nAnomalous toxicity cluster detected (Moving Average: {moving_avg:.2f} breached Dynamic Control Limit: {dynamic_ucl:.2f}). Please remain respectful."
                    await message.channel.send(warning_msg)
                    
                    try:
                        # Deduct reputation via DB
                        await db.update_trust_score(message.author.id, 15.0)
                    except:
                        pass
                    
                    # 🧹 INSTANT LOCAL MEMORY RESET:
                    # Flush the local list back to baseline zeros so the next apology message is completely safe!
                    channel_windows[message.channel.id] = [0.0, 0.0, 0.0, 0.0, 0.0]
                    print("Local channel window cache reset to baseline [0,0,0,0,0] to clear statistical hangover.")
                    
                    # Activate Cooldown
                    channel_cooldowns[message.channel.id] = time.time()
                    print("⏱️ Channel mitigation loop placed on a 60-second cooldown.")
                else:
                    print("⏳ Statistical anomaly ongoing, but channel mitigation is on cooldown.")
                    
    except Exception as core_err:
        print(f"❌ Critical Event Loop Crash Prevented: {core_err}")

    await bot.process_commands(message)

# Initialize the live runtime environment inside Jupyter
try:
    asyncio.create_task(bot.start(DISCORD_TOKEN))
except Exception as e:
    print(f"⚠️ Failed to initialize Aegis Core Engine: {e}")

✅ Aegis Agent Online: Logged in as DC Monitor#7557
--- AI Core & Real-Time Telemetry Loop Active ---
⚠️ Database Sync Delay/Failure (PostgreSQL Connection): duplicate key value violates unique constraint "messages_pkey"
DETAIL:  Key (message_id)=(1509567941799186583) already exists.
[Telemetry Scan] Score: 0.93 | sake3376: fucker


## 4. Results & SPC Validation
The autonomous agent was subjected to a simulated toxicity spiral to test the Statistical Process Control (SPC) intervention loop. 

* **LLM Labeling:** The NLP engine successfully categorized aggressive statements with high confidence (e.g., scores of 0.90 - 0.96).
* **Regex Extraction:** Implemented strict Regex parsing to prevent JSON decode errors from verbose LLM outputs.
* **Rolling Window:** The agent successfully calculated a 5-message moving average. Upon crossing the upper control limit (`> 0.60`), the bot autonomously issued a warning to the Discord channel and purged the local channel memory to prevent spam loops.

## 5. Conclusion & Future Production Scaling
This prototype successfully proves the viability of using open-weight LLMs combined with traditional statistical heuristics to manage digital communities. By relying on a moving average rather than single-message triggers, the system avoids over-policing and only intervenes during genuine toxicity spikes.

**Next Steps for Production (Silicon Valley Rigor):**
1. **Database Migration:** Swap the local SQLite prototype for a distributed `PostgreSQL` database to handle high-volume, multi-server concurrency.
2. **Containerization:** Package the bot using `Docker` to ensure environment consistency.
3. **Model Fine-Tuning:** Swap the generalized Llama 3 model for a custom LoRA-tuned model trained specifically on gaming/Discord-specific slang (e.g., bridging the "semantic drift" mentioned in the curriculum architecture).

In [ ]:
%%writefile dashboard.py
import streamlit as st
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

st.set_page_config(page_title="Aegis Admin", page_icon="🛡️", layout="wide")
st.title("🛡️ Aegis Community Manager: Admin Dashboard")
st.markdown("Real-time telemetry and Statistical Process Control for server health.")

# ---> PASTE YOUR SUPABASE URI HERE AS WELL <---
load_dotenv()
DATABASE_URL = os.getenv("SUPABASE_URL")
engine = create_engine(DATABASE_URL)

try:
    df_messages = pd.read_sql("SELECT * FROM messages", engine)
    df_users = pd.read_sql("SELECT * FROM users", engine)
    
    if df_messages.empty:
        st.warning("Database is empty. Go send some messages in Discord!")
    else:
        df_messages['timestamp'] = pd.to_datetime(df_messages['timestamp'])
        
        col1, col2, col3 = st.columns(3)
        col1.metric("Messages Scanned", len(df_messages))
        col2.metric("Global Avg Toxicity", f"{df_messages['toxicity_score'].mean():.2f}")
        highly_toxic = len(df_messages[df_messages['toxicity_score'] >= 0.60])
        col3.metric("Spikes Blocked (>0.60)", highly_toxic)
        
        st.markdown("---")
        
        st.subheader("📈 Server Health Timeline (Toxicity Score)")
        chart_data = df_messages[['timestamp', 'toxicity_score']].set_index('timestamp')
        st.line_chart(chart_data)
        
        col_left, col_right = st.columns(2)
        
        with col_left:
            st.subheader("⚠️ User Trust Scores")
            # Sort by lowest trust score
            leaderboard = df_users[['user_id', 'trust_score']].sort_values(by='trust_score', ascending=True).reset_index(drop=True)
            leaderboard.columns = ['Discord User ID', 'Current Trust Score']
            st.dataframe(leaderboard, use_container_width=True)
            
        with col_right:
            st.subheader("📝 Live Message Intercepts")
            recent_logs = df_messages[['timestamp', 'content', 'toxicity_score']].sort_values(by='timestamp', ascending=False)
            st.dataframe(recent_logs, use_container_width=True)

except Exception as e:
    st.error(f"Error loading database: {e}")

Writing dashboard.py
